# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Get a Python dict representation of the metadata for printing summary info
metadata_dict = dataset.metadata.to_json()
print(f"Name: {metadata_dict['name']}")
print(f"Description: {metadata_dict['description']}")
print(f"Published: {metadata_dict.get('datePublished', 'N/A')}")
print(f"Keywords: {metadata_dict.get('keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate the available record sets in the Croissant package and for each, print the `@id`, name, and field details. This provides an overview to help select fields for deeper analysis.

In [ ]:
# List record sets and their fields
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in this Croissant package.")
else:
    for record_set in record_sets:
        print(f"RecordSet name: {getattr(record_set, 'name', '(no name)')}")
        print(f"  @id: {record_set.id}")
        print(f"  Description: {getattr(record_set, 'description', '')}")
        print("  Fields and Columns:")
        for field in getattr(record_set, 'fields', []):
            print(f"    Field @id: {field.id}, Name: {getattr(field, 'name', '(no name)')}, Data type: {getattr(field, 'data_type', 'N/A')}")
        print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we extract all available record sets into Pandas DataFrames for convenient access. Replace the `selected_record_set_id` with the specific `@id` as needed.

In [ ]:
# Build a list of record_set @id values
record_set_ids = [r.id for r in dataset.record_sets]
dataframes = {}

# Iterate and extract each record set into a DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {rs_id} with columns: {dataframes[rs_id].columns.tolist()}")
    else:
        print(f"No records found for {rs_id}.")

# For demonstration, pick the first available DataFrame
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Selected record set for analysis: {selected_record_set_id}")
    print(dataframes[selected_record_set_id].head())
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations might include removing outliers, transforming values, or grouping.

In this section, we select a numeric field from the chosen record set and perform basic EDA.

In [ ]:
import numpy as np

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    # Try to auto-detect a numeric field
    candidate_numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if not candidate_numeric_fields:
        print("No numeric fields found. Trying to coerce columns to numeric to find possible candidates...")
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notna().sum() > 0 and coerced.notna().mean() > 0.5:
                df[col] = coerced
                candidate_numeric_fields.append(col)
        print(f"Numeric fields after coercion: {candidate_numeric_fields}")

    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]

        # Use threshold as ~median for illustration
        if df[numeric_field].dtype.kind in 'fi':
            threshold = df[numeric_field].median()
        else:
            threshold = 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a non-numeric field
        group_field = None
        for col in df.columns:
            if col != numeric_field and not np.issubdtype(df[col].dtype, np.number):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No suitable group field available.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No DataFrame extracted for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following example shows a histogram of the selected numeric field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and 'numeric_field' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Pairwise scatter if other numeric columns are available
    if len(candidate_numeric_fields) > 1:
        other_field = candidate_numeric_fields[1]
        plt.figure(figsize=(7, 5))
        plt.scatter(df[numeric_field], df[other_field], alpha=0.6)
        plt.xlabel(numeric_field)
        plt.ylabel(other_field)
        plt.title(f"{numeric_field} vs {other_field}")
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to:
- Load a Croissant dataset package with `mlcroissant`
- List available record sets and fields by their `@id`
- Extract and analyze records using Pandas
- Perform essential EDA and visualize numeric fields

By referring to all dataset entities by their `@id`, we maintain reproducibility and clarity. For deeper analysis, further data cleaning or domain-specific feature engineering may be needed, depending on the scientific goals.